# 05 — Xuất DATA cuối

Mỗi crop được sao chép nguyên pixel vào thư mục theo split và mệnh giá. Tên file là `<mệnh_giá>_<số thứ tự 6 chữ số>.png`; bộ đếm chạy liên tục theo từng mệnh giá sau khi sắp theo nguồn và đường dẫn gốc.

Notebook tạo thư mục staging có dấu thời gian, kèm `metadata/dataset.csv`. Trong metadata, `source_rel_path` tính từ dataset nguồn, `crop_path` từ `DATA_CROP`, và `final_path` từ `DATA`. Ô cuối chuyển `DATA` cũ sang thư mục backup bên cạnh project rồi kích hoạt staging.

In [1]:
from datetime import datetime
from pathlib import Path
import shutil
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "DATA_RAW").is_dir() else cwd.parents[1]
DATA_CROP = ROOT / "DATA_CROP"
DATA_DIR = ROOT / "DATA"
METADATA_DIR = DATA_CROP / "metadata"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
BUILD_DIR = ROOT / f"DATA_BUILD_05_{RUN_TIMESTAMP}"
splits = ["train", "dev", "test"]
denominations = ["10000", "20000", "50000", "100000", "200000", "500000"]
split_crops = pd.read_csv(METADATA_DIR / "split_crops.csv", dtype=str).fillna("")


In [2]:
BUILD_DIR.mkdir()
for split in splits:
    for denomination in denominations:
        (BUILD_DIR / split / denomination).mkdir(parents=True)
(BUILD_DIR / "metadata").mkdir()

ordered_crops = split_crops.sort_values(
    ["dataset_id", "source_rel_path", "crop_path"], kind="stable"
)
class_counters = {denomination: 0 for denomination in denominations}
metadata_rows = []
for row in ordered_crops.to_dict("records"):
    denomination = row["denomination"]
    class_counters[denomination] += 1
    final_filename = f"{denomination}_{class_counters[denomination]:06d}.png"
    final_path = Path(row["split"]) / denomination / final_filename
    shutil.copy2(DATA_CROP / row["crop_path"], BUILD_DIR / final_path)
    metadata_rows.append({
        "dataset_id": row["dataset_id"],
        "source_rel_path": row["source_rel_path"],
        "crop_path": row["crop_path"],
        "group_id": row["group_id"],
        "split": row["split"],
        "denomination": denomination,
        "final_filename": final_filename,
        "final_path": final_path.as_posix(),
    })

manifest = pd.DataFrame(metadata_rows)
manifest.to_csv(BUILD_DIR / "metadata" / "dataset.csv", index=False, encoding="utf-8-sig")


In [3]:
count_table = pd.crosstab(manifest["denomination"], manifest["split"])
count_table = count_table.reindex(index=denominations, columns=splits, fill_value=0)
count_table.loc["Tổng"] = count_table.sum()
display(count_table)


split,train,dev,test
denomination,,,
10000,2299,315,281
20000,1916,238,224
50000,1646,195,223
100000,1918,250,247
200000,2525,309,317
500000,2190,257,261
Tổng,12494,1564,1553


In [4]:
BACKUP_ROOT = ROOT.parent / f"{ROOT.name}_DATA_BACKUPS"
BACKUP_DIR = BACKUP_ROOT / f"DATA_BEFORE_05_{RUN_TIMESTAMP}"
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
if DATA_DIR.exists():
    DATA_DIR.rename(BACKUP_DIR)
BUILD_DIR.rename(DATA_DIR)


WindowsPath('C:/CS231/DATA')